# 철강 표면 결함 분류 — 최종 모델

누적 최적화 실험(`누적최적화실험.ipynb`)의 결론 모델만 정리한 노트북.

- **입력**: 원본(1채널) + Sobel 경계강도(1채널) = 2채널, train 셋 통계로 채널별 정규화
- **모델**: `DeepCNN` — Conv 3블록(채널 8-16-32, 3×3 padding=1, 각 Conv 뒤 BatchNorm) → FC
- **선택 근거**: E2(Sobel) 대비 파라미터 1/3.5 (≈15.4만) 로 동등한 Val Macro F1(≈0.94) 유지

## 0. 환경 설정

In [ ]:
import copy
import random
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("device:", device)

## 1. 데이터 로드 및 분할

In [ ]:
path = kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database")

data_root = Path(path) / "NEU-DET"
train_root = data_root / "train" / "images"
heldout_root = data_root / "validation" / "images"

class_names = sorted(folder.name for folder in train_root.iterdir() if folder.is_dir())
rng = np.random.default_rng(SEED)

train_samples, val_samples, test_samples = [], [], []
for class_id, class_name in enumerate(class_names):
    train_files = sorted((train_root / class_name).glob("*.jpg"))
    heldout_files = sorted((heldout_root / class_name).glob("*.jpg"))
    order = rng.permutation(len(heldout_files))

    val_samples.extend((heldout_files[i], class_id) for i in order[:30])
    test_samples.extend((heldout_files[i], class_id) for i in order[30:])
    train_samples.extend((file, class_id) for file in train_files)

print("classes:", class_names)
print("split  :", len(train_samples), len(val_samples), len(test_samples))

In [ ]:
class SteelDefectDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        file, label = self.samples[index]
        image = Image.open(file).convert("L")
        return self.transform(image), label

## 2. 전처리 — Sobel 경계 채널 + 정규화

정규화 전 텐서 `(1,H,W)` 에 Sobel 경계강도 채널을 붙여 `(2,H,W)` 로 만들고,
train 셋에서 계산한 채널별 mean/std 로 정규화한다 (val/test 누수 방지).

In [ ]:
# Sobel 커널 (학습되지 않는 고정 필터)
SOBEL_X = torch.tensor([[-1.0, 0.0, 1.0],
                        [-2.0, 0.0, 2.0],
                        [-1.0, 0.0, 1.0]]).view(1, 1, 3, 3)
SOBEL_Y = torch.tensor([[-1.0, -2.0, -1.0],
                        [0.0, 0.0, 0.0],
                        [1.0, 2.0, 1.0]]).view(1, 1, 3, 3)


def add_sobel_channel(img):
    """img: (1, H, W) float in [0, 1]  ->  (2, H, W) = [원본, 경계강도]"""
    x = img.unsqueeze(0)  # (1, 1, H, W)
    gx = F.conv2d(x, SOBEL_X, padding=1)
    gy = F.conv2d(x, SOBEL_Y, padding=1)
    edge = torch.sqrt(gx ** 2 + gy ** 2).squeeze(0)  # (1, H, W)
    return torch.cat([img, edge], dim=0)  # (2, H, W)


def build_transforms(sobel=False, normalize=None):
    ops = [transforms.Resize((96, 96)), transforms.ToTensor()]
    if sobel:
        ops.append(transforms.Lambda(add_sobel_channel))
    if normalize is not None:
        ops.append(transforms.Normalize(*normalize))
    return transforms.Compose(ops)


def compute_channel_stats(transform, channels):
    """train 셋에서 채널별 mean/std 계산."""
    loader = DataLoader(SteelDefectDataset(train_samples, transform), batch_size=64)
    total = torch.zeros(channels)
    total_sq = torch.zeros(channels)
    count = 0
    for images, _ in loader:  # (B, C, H, W)
        total += images.sum(dim=[0, 2, 3])
        total_sq += (images ** 2).sum(dim=[0, 2, 3])
        count += images.shape[0] * images.shape[2] * images.shape[3]
    mean = total / count
    std = (total_sq / count - mean ** 2).sqrt()
    return tuple(mean.tolist()), tuple(std.tolist())

In [ ]:
NORM_2CH = compute_channel_stats(build_transforms(sobel=True), channels=2)
final_tf = build_transforms(sobel=True, normalize=NORM_2CH)

train_loader = DataLoader(SteelDefectDataset(train_samples, final_tf), batch_size=64, shuffle=True)
val_loader = DataLoader(SteelDefectDataset(val_samples, final_tf), batch_size=64)
test_loader = DataLoader(SteelDefectDataset(test_samples, final_tf), batch_size=64)

print("channel mean/std:", NORM_2CH)

## 3. 학습 / 평가 헬퍼

In [ ]:
loss_fn = nn.CrossEntropyLoss()


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = correct = count = 0

    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            count += len(images)

    return total_loss / count, correct / count


def predict(model, loader):
    model.eval()
    answers, preds = [], []
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(device))
            preds.extend(outputs.argmax(1).cpu().numpy())
            answers.extend(labels.numpy())
    return np.asarray(answers), np.asarray(preds)


def _to_disp(image):
    """(C,H,W) 텐서를 흑백 표시용 2D로. 2채널이면 원본(0번) 채널 사용."""
    if image.dim() == 3 and image.size(0) > 1:
        return image[0]
    return image.squeeze()


def plot_history(history, tag):
    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], marker="o", label="train")
    plt.plot(epochs, history["val_loss"], marker="o", label="val")
    plt.title(f"{tag} - Loss")
    plt.xlabel("epoch")
    plt.grid(alpha=0.3)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], marker="o", label="train acc")
    plt.plot(epochs, history["val_acc"], marker="o", label="val acc")
    plt.plot(epochs, history["val_f1"], marker="o", label="val f1")
    plt.title(f"{tag} - Accuracy / F1")
    plt.xlabel("epoch")
    plt.ylim(0, 1.05)
    plt.grid(alpha=0.3)
    plt.legend()

    plt.tight_layout()
    plt.show()


def show_confusion(answers, preds, title):
    ConfusionMatrixDisplay.from_predictions(
        answers, preds, display_labels=class_names, cmap="Blues", xticks_rotation=35,
    )
    plt.title(title)
    plt.tight_layout()
    plt.show()


def show_predictions(model, dataset, n_per_class=2, title="Predictions"):
    model.eval()
    picks = []
    for class_id in range(len(class_names)):
        picks += [i for i, (_, label) in enumerate(dataset.samples) if label == class_id][:n_per_class]

    cols = n_per_class * 2
    rows = (len(picks) + cols - 1) // cols
    plt.figure(figsize=(3 * cols, 3 * rows))
    for position, data_index in enumerate(picks, start=1):
        image, answer = dataset[data_index]
        with torch.no_grad():
            pred = model(image.unsqueeze(0).to(device)).argmax(1).item()
        plt.subplot(rows, cols, position)
        plt.imshow(_to_disp(image), cmap="gray")
        color = "black" if pred == answer else "red"
        plt.title(f"T: {class_names[answer]}\nP: {class_names[pred]}", color=color, fontsize=9)
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

## 4. 최종 모델 — DeepCNN

Conv 3블록 (3×3, padding=1 → 크기 유지) + 각 Conv 뒤 BatchNorm → FC.
`96 → 48 → 24 → 12`, 채널 진행 `8 → 16 → 32`, FC 입력 `32·12·12 = 4608` (dummy forward 로 자동 계산).

In [ ]:
class DeepCNN(nn.Module):
    def __init__(self, in_channels=2, use_bn=True, channels=(8, 16, 32)):
        super().__init__()
        blocks = []
        prev = in_channels
        for channel in channels:
            blocks.append(nn.Conv2d(prev, channel, kernel_size=3, padding=1))
            if use_bn:
                blocks.append(nn.BatchNorm2d(channel))
            blocks.append(nn.ReLU())
            blocks.append(nn.MaxPool2d(kernel_size=2))
            prev = channel
        self.features = nn.Sequential(*blocks)

        with torch.no_grad():
            flat = self.features(torch.zeros(1, in_channels, 96, 96)).flatten(1).shape[1]

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat, 32),
            nn.ReLU(),
            nn.Linear(32, 6),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


print(f"parameters: {sum(p.numel() for p in DeepCNN().parameters()):,}")

## 5. 학습

In [ ]:
def train_final(build_model, train_loader, val_loader, *, tag="final_deepcnn", epochs=15, lr=1e-3):
    set_seed(SEED)
    model = build_model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {k: [] for k in ["train_loss", "val_loss", "train_acc", "val_acc", "val_f1"]}
    best_val_loss, best_state = float("inf"), None

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader)
        val_answers, val_preds = predict(model, val_loader)
        val_f1 = f1_score(val_answers, val_preds, average="macro")

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)

        if val_loss < best_val_loss:  # val_loss 최소 시점 복원
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch:02d} | train {train_loss:.3f}/{train_acc:.3f} | "
            f"val {val_loss:.3f}/{val_acc:.3f} | val_f1 {val_f1:.3f}"
        )

    model.load_state_dict(best_state)
    Path("weights_report").mkdir(exist_ok=True)
    torch.save(best_state, f"weights_report/{tag}.pt")
    return model, history


model, history = train_final(lambda: DeepCNN(in_channels=2, use_bn=True), train_loader, val_loader)
plot_history(history, "final_deepcnn")

## 6. Validation 진단

In [ ]:
val_answers, val_preds = predict(model, val_loader)
print(f"Val Accuracy = {accuracy_score(val_answers, val_preds):.4f}")
print(f"Val Macro F1 = {f1_score(val_answers, val_preds, average='macro'):.4f}\n")
print(classification_report(val_answers, val_preds, target_names=class_names))
show_confusion(val_answers, val_preds, "Final - Validation Confusion Matrix")
show_predictions(model, val_loader.dataset, title="Final - Validation predictions")

## 7. 테스트셋 최종 평가

학습·모델선택에 전혀 쓰지 않은 test 셋으로 한 번만 평가한다.

In [ ]:
test_answers, test_preds = predict(model, test_loader)
test_acc = accuracy_score(test_answers, test_preds)
test_f1 = f1_score(test_answers, test_preds, average="macro")

print(f"Test Accuracy = {test_acc:.4f}")
print(f"Test Macro F1 = {test_f1:.4f}\n")
print(classification_report(test_answers, test_preds, target_names=class_names))
show_confusion(test_answers, test_preds, "Final - Test Confusion Matrix")
show_predictions(model, test_loader.dataset, title="Final - Test predictions")